환경변수

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

라이브러리

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph
from langgraph.graph import END
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

LLM

In [ ]:
llm = ChatOpenAI(
    model="gpt-5.4-nano",
    temperature=0
)

embedding = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

Vector DB 연결

In [ ]:
law_db = Chroma(
    persist_directory="vector_db/laws",
    embedding_function=embedding
)

precedent_db = Chroma(
    persist_directory="vector_db/precedents",
    embedding_function=embedding
)

State

In [ ]:
class GraphState(TypedDict):
    
    question: str

    law_docs: list
    law_analysis: str

    precedent_docs: list
    precedent_analysis: str

    final_answer: str

법령 검색 노드

In [ ]:
def retrieve_law_node(state):

    question = state["question"]

    docs = law_db.similarity_search(
        question,
        k=5
    )

    print("\n===== LAW RETRIEVAL TOP5 =====")
    for i, doc in enumerate(docs, 1):
        print(f"\n[LAW {i}]")
        print("metadata:", doc.metadata)
        print(doc.page_content[:500])  # 앞 500자만 출력

    return {
        "law_docs": docs
    }

법령 분석 노드

In [ ]:
def analyze_law_node(state):

    question = state["question"]

    # 조문 단위 메타데이터로 출처 라벨 구성
    law_context_parts = []
    for doc in state["law_docs"]:
        m = doc.metadata
        law_name   = m.get("law_name", "")
        article_no = m.get("article_no", "")
        article_title = m.get("article_title", "")
        chapter    = m.get("chapter", "")
        source_pdf = m.get("source_pdf", "")
        page       = m.get("page", "")

        # "근로기준법 / 제3장 제43조 (임금 지급) / 1p" 형태
        parts = [p for p in [law_name, chapter, f"{article_no} ({article_title})" if article_title else article_no] if p]
        label = " / ".join(parts)
        if page:
            label += f" / {page}p"
        if source_pdf:
            label += f" [{source_pdf}]"

        law_context_parts.append(f"[출처: {label}]\n{doc.page_content}")

    law_context = "\n\n".join(law_context_parts)

    prompt = f"""
당신은 대한민국 노동법 전문 AI입니다.

사용자 사례:
{question}

관련 법령 (각 조문 앞에 출처 표시):
{law_context}

규칙

1. 관련 법률만 설명
2. 위법이라고 단정 금지
3. 반드시 법률명, 장(章), 조(條), 항(項)을 명시 (예: 근로기준법 제3장 제43조 제1항)
4. 조문이 확인되지 않으면 "조문 불명확"으로 표시
5. 출처 PDF 파일명도 함께 표시

출력 형식 (반드시 준수)

관련 법률:
- 법률명 제X장 제X조 제X항 (출처: [파일명] Xp): 조항 내용 요약

설명:
- 해당 조항이 사용자 사례에 적용되는 이유
"""

    result = llm.invoke(prompt).content

    return {
        "law_analysis": result
    }


판례 검색 노드

In [ ]:
def retrieve_precedent_node(state):

    keyword_prompt = f"""
다음 사용자 사례와 법률 분석에서
판례 검색에 사용할 핵심 법률 쟁점 키워드를
10단어 이내로 추출하세요.

사용자 사례: {state['question']}
법률 분석 요약: {state['law_analysis'][:300]}

출력: 키워드만, 문장 금지
"""

    search_query = llm.invoke(keyword_prompt).content.strip()

    print("\n===== PRECEDENT SEARCH QUERY =====")
    print(search_query)

    docs = precedent_db.similarity_search(
        search_query,
        k=5
    )

    print("\n===== PRECEDENT RETRIEVAL TOP5 =====")
    for i, doc in enumerate(docs, 1):
        print(f"\n[PRECEDENT {i}]")
        print("metadata:", doc.metadata)
        print(doc.page_content[:500])

    return {
        "precedent_docs": docs
    }

판례 분석 노드

In [ ]:
def analyze_precedent_node(state):

    # 판례 메타데이터: source_file (예: "2011다23149.md")
    precedent_context_parts = []
    for doc in state["precedent_docs"]:
        m = doc.metadata
        source_file = m.get("source_file", "알 수 없는 판례")
        # 파일명에서 사건번호 추출 (확장자 제거)
        case_no = source_file.replace(".md", "").replace(".json", "")
        label = f"[판례: {case_no}]"
        precedent_context_parts.append(f"{label}\n{doc.page_content}")

    precedent_context = "\n\n".join(precedent_context_parts)

    prompt = f"""
사용자 사례:
{state['question']}

검색된 판례 (각 판례 앞에 출처 표시):
{precedent_context}

규칙

1. 반드시 판례명(사건번호) 전체 표기 (예: 대법원 2019. 4. 18. 선고 2016다2451 판결)
2. 판결 요지 1~2문장으로 핵심 요약
3. 사용자 사례와의 유사점을 구체적으로 설명
4. 법률 자문처럼 단정 금지
5. 출처 파일명(사건번호) 반드시 표시

출력 형식 (반드시 준수)

판례명: [전체 사건번호 포함]
출처: [파일명]
판결 요지: [핵심 내용 1~2문장]
사용자 사례 관련성: [구체적 유사점]
"""

    result = llm.invoke(prompt).content

    return {
        "precedent_analysis": result
    }


최종 답변 노드

In [ ]:
def generate_answer_node(state):

    prompt = f"""
사용자 사례:
{state["question"]}

법률 분석:
{state["law_analysis"]}

판례 분석:
{state["precedent_analysis"]}

당신은 노동법 분석 시스템이다.

반드시 아래 형식만 출력하라.

# 관련 법률

- 법률명 및 조항: 예) 근로기준법 제3장 제43조 제1항
- 조항 내용 요약
- 적용 가능성

# 관련 판례

- 판례명: 예) 대법원 2019. 4. 18. 선고 2016다2451 판결
- 출처: 판례 파일명 / 분류
- 판결 요지
- 사용자 사례와의 관련성

# 결론

- 현재 정보 기준 분석 결과
- 주요 근거 조항 재정리 (법률명 + 조번호)

규칙

1. 추가 질문 금지
2. 추가 정보 요청 금지
3. 상담 권유 금지
4. "원하시면", "추가로 알려주시면" 같은 문장 금지
5. 위 형식 외 내용 출력 금지
6. 최종 결과만 작성
7. 조항 번호가 불명확한 경우 "조항 확인 필요"로 표시
"""

    answer = llm.invoke(prompt).content

    return {
        "final_answer": answer
    }


Graph 생성

In [ ]:
builder = StateGraph(GraphState)

builder.add_node(
    "retrieve_law",
    retrieve_law_node
)

builder.add_node(
    "analyze_law",
    analyze_law_node
)

builder.add_node(
    "retrieve_precedent",
    retrieve_precedent_node
)

builder.add_node(
    "analyze_precedent",
    analyze_precedent_node
)

builder.add_node(
    "generate_answer",
    generate_answer_node
)

Edge 연결

In [ ]:
builder.set_entry_point(
    "retrieve_law"
)

builder.add_edge(
    "retrieve_law",
    "analyze_law"
)

builder.add_edge(
    "analyze_law",
    "retrieve_precedent"
)

builder.add_edge(
    "retrieve_precedent",
    "analyze_precedent"
)

builder.add_edge(
    "analyze_precedent",
    "generate_answer"
)

builder.add_edge(
    "generate_answer",
    END
)

graph = builder.compile()

Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.
Adding an edge to a graph that has already been compiled. This will not be reflected in the compiled graph.


법령 PDF 벡터DB 생성 (조문 단위 파싱 — law_parser.py 사용)

In [ ]:
import json
from pathlib import Path

from langchain_core.documents import Document
from langchain_chroma import Chroma

# ==========================================================
# 법령 JSON 로드
# ==========================================================

law_root = Path("data/process/law")

if not law_root.exists():
    raise FileNotFoundError(
        f"폴더가 없습니다: {law_root.resolve()}\n"
        "law_parser.py의 process_all_pdfs()를 먼저 실행해 주세요."
    )

json_files = sorted(law_root.rglob("*.json"))

if not json_files:
    raise FileNotFoundError(
        f"{law_root} 안에 JSON 파일이 없습니다.\n"
        "law_parser.py의 process_all_pdfs()를 먼저 실행해 주세요."
    )

print(f"JSON 파일 {len(json_files)}개 발견")

# ==========================================================
# JSON → Document
# ==========================================================

all_law_docs = []

for json_file in json_files:

    print(f"로딩 중: {json_file.name}")

    with open(json_file, "r", encoding="utf-8") as f:
        items = json.load(f)

    if isinstance(items, dict):
        items = [items]

    for item in items:

        meta = item.get("metadata", {})

        chapter_title = meta.get("chapter_title", "")
        article_title = meta.get("article_title", "")
        page_content = item.get("page_content", "")

        # --------------------------------------------------
        # 임베딩용 텍스트
        # chapter_title
        # article_title
        # page_content
        # --------------------------------------------------

        embedding_text = f"""
{chapter_title}

{article_title}

{page_content}
"""

        embedding_text = " ".join(
            embedding_text.split()
        )

        all_law_docs.append(
            Document(
                page_content=embedding_text,
                metadata=meta
            )
        )

if not all_law_docs:
    raise ValueError(
        "로드된 법령 Document가 0개입니다."
    )

print(f"\n총 {len(all_law_docs)}개 법령 Document 로드 완료")

print("\n========== Document 예시 ==========")
print(all_law_docs[0].page_content[:500])

print("\n========== Metadata 예시 ==========")
for k, v in all_law_docs[0].metadata.items():
    print(f"{k}: {v}")

# ==========================================================
# Chroma DB 생성
# ==========================================================

law_db = Chroma.from_documents(
    documents=all_law_docs,
    embedding=embedding,
    persist_directory="vector_db/laws"
)

print("\n법령 DB 생성 완료")
print("저장 위치: vector_db/laws")

JSON 파일 24개 발견
로딩 중: 근로기준법 시행규칙(고용노동부령)(제00436호)(20250223).json
로딩 중: 근로기준법 시행령(대통령령)(제35436호)(20251023).json
로딩 중: 근로기준법(법률)(제20520호)(20251023) (1).json
로딩 중: 근로자퇴직급여 보장법 시행규칙(고용노동부령)(제00359호)(20220712).json
로딩 중: 근로자퇴직급여 보장법 시행령(대통령령)(제36220호)(20260324).json
로딩 중: 근로자퇴직급여 보장법(법률)(제21135호)(20251111).json
로딩 중: 기간제 및 단시간근로자 보호 등에 관한 법률 시행규칙(노동부령)(제00277호)(20070701).json
로딩 중: 기간제 및 단시간근로자 보호 등에 관한 법률 시행령(대통령령)(제31611호)(20210408).json
로딩 중: 기간제 및 단시간근로자 보호 등에 관한 법률(법률)(제18177호)(20210518).json
로딩 중: 남녀고용평등과 일ㆍ가정 양립 지원에 관한 법률 시행규칙(고용노동부령)(제00453호)(20251001).json
로딩 중: 남녀고용평등과 일ㆍ가정 양립 지원에 관한 법률 시행령(대통령령)(제35806호)(20251001).json
로딩 중: 남녀고용평등과 일ㆍ가정 양립 지원에 관한 법률(법률)(제21065호)(20251001).json
로딩 중: 노동조합 및 노동관계조정법 시행규칙(고용노동부령)(제00463호)(20260310).json
로딩 중: 노동조합 및 노동관계조정법 시행령(대통령령)(제36159호)(20260310).json
로딩 중: 노동조합 및 노동관계조정법(법률)(제21045호)(20260310).json
로딩 중: 산업안전보건법 시행규칙(고용노동부령)(제00470호)(20260601).json
로딩 중: 산업안전보건법 시행령(대통령령)(제36220호)(20260324).json
로딩 중: 산업안전보건법(법률)(제21374호)(20260601).json
로딩 중:

판례 벡터DB 생성

In [ ]:
import json
from pathlib import Path
from langchain_core.documents import Document
from langchain_chroma import Chroma

precedent_root = Path("data/process/판례")
all_docs = []

# JSON 파일로 저장된 판례 로드
for json_file in sorted(precedent_root.rglob("*.json")):

    print(f"로딩 중: {json_file.name}")

    with open(json_file, "r", encoding="utf-8") as f:
        items = json.load(f)

    # 파일 하나에 판례 여러 건이 리스트로 담겨 있을 수 있음
    if isinstance(items, dict):
        items = [items]

    for item in items:
        doc = Document(
            page_content=item["page_content"],
            metadata=item.get("metadata", {})
        )
        all_docs.append(doc)

print(f"\n총 {len(all_docs)}개 판례 Document 로드 완료")
if all_docs:
    print("[메타데이터 예시]", all_docs[0].metadata)

# 판례는 1건이 하나의 Document — 청크 분할 없음
precedent_db = Chroma.from_documents(
    documents=all_docs,
    embedding=embedding,
    persist_directory="vector_db/precedents"
)

print("판례 DB 생성 완료")

로딩 중: 2011다23149.json
로딩 중: 2006다64245.json
로딩 중: 2014다44673.json
로딩 중: 2018도6486.json
로딩 중: 2001도204.json
로딩 중: 2017도4343.json
로딩 중: 2022도2188.json
로딩 중: 98도1260.json
로딩 중: 2010다91046.json
로딩 중: 2012다89399.json
로딩 중: 2015다73067.json
로딩 중: 89다카2292.json
로딩 중: 94다19501.json
로딩 중: 2022두64518.json
로딩 중: 90누2772.json
로딩 중: 97다5015.json
로딩 중: 2012다106423.json
로딩 중: 2015다73067.json
로딩 중: 2020도15393.json
로딩 중: 2008다57852.json
로딩 중: 2008다6052.json
로딩 중: 96다24699.json
로딩 중: 2007다90760.json
로딩 중: 2016다255910.json
로딩 중: 93다26168.json
로딩 중: 97누18189.json
로딩 중: 2002다11458.json
로딩 중: 90다11554.json
로딩 중: 91다36666.json
로딩 중: 2020도16541.json
로딩 중: 2004다29736.json
로딩 중: 2006도777.json
로딩 중: 2009다51417.json
로딩 중: 2002다62432.json
로딩 중: 2019두55859.json
로딩 중: 2006두4912.json
로딩 중: 2017두45933.json
로딩 중: 2019두62604.json
로딩 중: 2020다270503.json
로딩 중: 2007두22498.json
로딩 중: 2017두74702.json
로딩 중: 2012두4852.json
로딩 중: 2015두51651.json
로딩 중: 2017두76005.json
로딩 중: 2018두47264.json
로딩 중: 2019두38571.json

총 46개 판례 Document

In [ ]:
result = graph.invoke(
    {
        "question":
        """근로자가 국민참여재판의 배심원으로 참석하는 시간을 유급으로 처리해야 
하는지"""
    }
)

print(result["final_answer"])


===== LAW RETRIEVAL TOP5 =====

[LAW 1]
metadata: {'law_name': '근로기준법', 'article_title': '특별한 사정이 있는 경우의 근로시간 연장 신청 등', 'article_no': '제9조', 'chapter_title': '', 'chapter_no': ''}
특별한 사정이 있는 경우의 근로시간 연장 신청 등 ① 법 제53조제4항 본문에서 “특별한 사정”이란 다음 각 호 의 어느 하나에 해당하는 경우를 말한다. 1. 「재난 및 안전관리 기본법」에 따른 재난 또는 이에 준하는 사고가 발생하여 이를 수습하거나 재난 등의 발생이 예상되어 이를 예방하기 위해 긴급한 조치가 필요한 경우 2. 사람의 생명을 보호하거나 안전을 확보하기 위해 긴급한 조치가 필요한 경우 3. 갑작스런 시설ㆍ설비의 장애ㆍ고장 등 돌발적인 상황이 발생하여 이를 수습하기 위해 긴급한 조치가 필요한 경 우 4. 통상적인 경우에 비해 업무량이 대폭적으로 증가한 경우로서 이를 단기간 내에 처리하지 않으면 사업에 중대한 지장을 초래하거나 손해가 발생하는 경우 5. 「소재ㆍ부품ㆍ장비산업 경쟁력 강화 및 공급망 안정화를 위한 특별조치법」 제2조제1호 및 제2호에 따른 소재ㆍ 부품 및 장비의 연구개발 등 연구개발을 하는 경우로서 고용노동부장관이 국가경쟁력 강화 및 국민경제 발전을 위해 필

[LAW 2]
metadata: {'chapter_no': '제2장', 'chapter_title': '근로계약', 'article_no': '제25조', 'article_title': '우선 재고용 등', 'law_name': '근로기준법'}
근로계약 우선 재고용 등 ① 제24조에 따라 근로자를 해고한 사용자는 근로자를 해고한 날부터 3년 이내에 해고된 근로 자가 해고 당시 담당하였던 업무와 같은 업무를 할 근로자를 채용하려고 할 경우 제24조에 따라 해고된 근로자가 원 하면 그 근로자를 우선적으로 고용하여야 한다. ② 정부는 제24조에 따라 해고된 근로자에 

In [ ]:
from IPython.display import Markdown, display

display(
    Markdown(result["final_answer"])
)

# 관련 법률

- 법률명 및 조항: 근로기준법 제10조 제1항  
- 조항 내용 요약: 사용자는 근로자가 **근로시간 중에 선거권 등 공민권 행사** 또는 **공의 직무를 집행**하기 위해 필요한 시간을 청구하면 이를 거부하지 못함.  
- 적용 가능성: 국민참여재판 배심원 참여가 근로자 입장에서 **‘공의 직무를 집행’**에 해당하여 “필요한 시간”이라 평가되면, 해당 시간에 대한 청구를 사용자가 거부할 수 없다는 방향으로 적용될 수 있음. 다만 **‘유급(임금 지급)’ 그 자체를 명시하는 규정은 현재 제시 목록에 없음**.

- 법률명 및 조항: 근로기준법 제9조 제1항~제4항 (조항 확인 필요)  
- 조항 내용 요약: 특별한 사정이 있으면 근로시간 연장에 대한 인가/승인 절차를 두고, 연장 가능 기간은 **필요 최소한**으로 함.  
- 적용 가능성: 배심원 참여시간을 **유급으로 처리할지(임금 지급 여부)**의 직접 근거로 보기 어려움. 다만 배심원 참여가 근로시간 운영(연장/휴게 등)과 연계되는 실무 쟁점이 있으면 간접 참고 가능.

- 법률명 및 조항: 근로기준법 제25조 제1항~제2항  
- 조항 내용 요약: 해고 근로자에 대한 우선 재고용 등 제도 취지 규정(유급 처리 직접 조문과는 거리가 있음).  
- 적용 가능성: 배심원 참여시간 **유급 처리**와 직접 관련성 낮음.

- 법률명 및 조항: 기간제 및 단기근로자 보호 등에 관한 법률 제20조  
- 조항 내용 요약: 기간제/단시간근로자 취업촉진을 위한 국가 등의 노력에 관한 규정.  
- 적용 가능성: 배심원 참여시간 **유급 처리**와 직접 관련성 낮음.

- 법률명 및 조항: 산업안전보건법 제27조  
- 조항 내용 요약: 안전보건교육 면제 요건.  
- 적용 가능성: 배심원 참여시간 **유급 처리**와 직접 관련성 낮음.

# 관련 판례

- 판례명: 대법원 2021. 3. 18. 선고 2018두47264 전원합의체 판결  
- 출처: 2018두47264.txt  
- 판결 요지: 사회보장수급권에서 ‘신청기간’이 권리행사의 조기 확정·강제를 위한 강행규정(제척기간)인지 여부를 판단함. 훈시규정이 아니라 강행규정이면 기간 경과 시 신청을 통한 권리행사가 제한될 수 있음.  
- 사용자 사례와의 관련성: 배심원 참여시간 ‘유급 처리’가 제도상 **청구/지급 요건과 연계**되어 있다면, 관련 규정에서 기간이 강행적으로 작동하는지(기간 경과의 불이익 발생 여부) 판단틀로 참고될 수 있음. 다만 이 판례는 **육아휴직급여 신청기간** 성격을 다룬 것으로, 배심원 유급 결론으로 직접 연결되지는 않음.

- 판례명: 대법원 2020. 5. 28. 선고 2019두62604 판결  
- 출처: 2019두62604.txt  
- 판결 요지: 업무상 재해 인정에서 인과관계는 반드시 의학적으로 명백할 필요는 없고 제반 사정으로 상당인과관계를 추단할 수 있으면 인정될 수 있음.  
- 사용자 사례와의 관련성: 배심원 참여에 따른 임금상 손실 보전이 문제되는 경우, 비용/손실 범위 산정이나 상당성 논증에 참고될 여지는 있으나 **유급 처리 여부 자체**의 직접 판단기준은 아님.

- 판례명: 대법원 2014. 6. 12. 선고 2012두4852 판결  
- 출처: 2012두4852.txt  
- 판결 요지: 육아휴직 복직 요건은 형식적 사유에 한정되지 않고, 실제로 양육 필요가 소멸했는지 등 실질을 고려해야 하며, 복직명령은 요건 충족 시 기속행위로 봄.  
- 사용자 사례와의 관련성: ‘유급 처리’가 되는 요건이 법상/절차상 어떻게 구성되어 있는지에서 **형식이 아니라 실질 요건**을 보는 논증이 가능할 여지는 있으나, 배심원 임금 지급 규정의 직접 해석 판례로 보기에는 한계가 있음.

- 판례명: 대법원 2022. 6. 30. 선고 2017두76005 판결  
- 출처: 2017두76005.txt  
- 판결 요지: 육아휴직 후 불리한 처우는 휴직 전후 근로조건 전반의 **실질적 불이익**을 의미하며, 같은 업무 해당성도 사회통념상 비교 기준으로 판단함.  
- 사용자 사례와의 관련성: 배심원 참여로 인한 임금/근로조건 불이익 여부가 문제될 때 **실질적 불이익** 관점으로 참고 가능. 다만 육아휴직 관련 법리라 배심원 유급 처리의 직접 결론 근거가 되기 어렵다.

- 판례명: 대법원 2023. 12. 7. 선고 2020도15393 판결  
- 출처: 2020도15393.txt  
- 판결 요지: 연장근로 12시간 초과 여부는 주 단위로 판단하는 등 근로시간 계산 기준을 명확히 함.  
- 사용자 사례와의 관련성: 배심원 참여시간이 근로시간으로 산정·관리되는 과정에서 “시간 계산 기준(주 단위 등)” 논증에 참고될 여지는 있으나, **유급 지급 여부**를 직접 판단하는 판례는 아님.

# 결론

- 현재 정보 기준 분석 결과:  
  - 배심원 참여시간이 근로시간 중 **공민권 행사/공의 직무 집행**에 해당한다면, 근로기준법 제10조 제1항에 의해 사용자는 해당 시간에 대한 청구를 **거부할 수 없는 방향**이 도출됨.  
  - 그러나 제시된 법률 목록(조문)만으로는 배심원 참여시간을 **임금으로 지급(유급 처리)해야 하는지에 대한 직접 근거 조문을 확인할 수 없으므로**, “유급으로 처리해야 한다/유급이 아니다”를 단정할 수 없음.

- 주요 근거 조항 재정리 (법률명 + 조번호)
  - 근로기준법 제10조 제1항 (공민권 행사/공의 직무 집행 필요시간 청구 거부 제한)  
  - 근로기준법 제9조 제1항~제4항: 조항 확인 필요 (근로시간 연장 관련 절차/범위로서 유급 여부 직접 근거 부족)

In [ ]:
docs = law_db.similarity_search(
    "사장이 저를 때렸습니다",
    k=5
)

for i, doc in enumerate(docs, 1):

    m = doc.metadata

    ordered_meta = {
        "law_name": m.get("law_name", ""),
        "chapter_no": m.get("chapter_no", ""),
        "chapter_title": m.get("chapter_title", ""),
        "article_no": m.get("article_no", ""),
        "article_title": m.get("article_title", ""),
        "section_no": m.get("section_no", ""),
        "section_title": m.get("section_title", ""),
    }

    print(f"\n[LAW {i}]")
    print("metadata:", ordered_meta)
    print("content:", doc.page_content[:500])


[LAW 1]
metadata: {'law_name': '남녀고용평등과 일가정 양립 지원에 관한 법률', 'chapter_no': '제3장', 'chapter_title': '모성 보호', 'article_no': '제22조의4', 'article_title': '가족돌봄 등을 위한 근로시간 단축 중 근로조건 등', 'section_no': '', 'section_title': ''}
content: 모성 보호 가족돌봄 등을 위한 근로시간 단축 중 근로조건 등 ① 사업주는 제22조의3에 따라 근로시간 단축을 하고 있는 근로자에게 근로시간에 비례하여 적용하는 경우 외에는 가족돌봄 등을 위한 근로시간 단축을 이유로 그 근로 조건을 불리하게 하여서는 아니 된다. ② 제22조의3에 따라 근로시간 단축을 한 근로자의 근로조건(근로시간 단축 후 근로시간을 포함한다)은 사업주와 그 근로자 간에 서면으로 정한다. ③ 사업주는 제22조의3에 따라 근로시간 단축을 하고 있는 근로자에게 단축된 근로시간 외에 연장근로를 요구할 수 없다. 다만, 그 근로자가 명시적으로 청구하는 경우에는 사업주는 주 12시간 이내에서 연장근로를 시킬 수 있다. ④ 근로시간 단축을 한 근로자에 대하여 「근로기준법」 제2조제6호에 따른 평균임금을 산정하는 경우에는 그 근로 자의 근로시간 단축 기간을 평균임금 산정기간에서 제외한다.

[LAW 2]
metadata: {'law_name': '남녀고용평등과 일가정 양립 지원에 관한 법률', 'chapter_no': '제3장', 'chapter_title': '모성 보호', 'article_no': '제22조의3', 'article_title': '가족돌봄 등을 위한 근로시간 단축', 'section_no': '', 'section_title': ''}
content: 모성 보호 가족돌봄 등을 위한 근로시간 단축 ① 사업주는 근로자가 다음 각 호의 어느 하나에 해당하는 사유로 근로 시간의 단축을 신청하는 경우에 이를 허용하여야 한다. 다만, 대체인력 채용이 

In [ ]:
docs = law_db.similarity_search(
    "근로자 폭행",
    k=10
)

for i, d in enumerate(docs, 1):
    print(f"\n=== {i} ===")
    print("metadata:", d.metadata)
    print("content:", d.page_content[:500])


=== 1 ===
metadata: {'article_no': '제7조', 'law_name': '근로기준법', 'article_title': '강제 근로의 금지', 'chapter_no': '제1장', 'chapter_title': '총칙'}
content: 총칙 강제 근로의 금지 사용자는 폭행, 협박, 감금, 그 밖에 정신상 또는 신체상의 자유를 부당하게 구속하는 수단으로 써 근로자의 자유의사에 어긋나는 근로를 강요하지 못한다. 근로기준법

=== 2 ===
metadata: {'chapter_no': '제2장', 'chapter_title': '근로자파견사업의 적정 운영', 'article_no': '제16조', 'article_title': '근로자파견의 제한', 'law_name': '파견근로자 보호 등에 관한 법률'}
content: 근로자파견사업의 적정 운영 근로자파견의 제한 ① 파견사업주는 쟁의행위 중인 사업장에 그 쟁의행위로 중단된 업무의 수행을 위하여 근 로자를 파견하여서는 아니 된다. ② 누구든지 「근로기준법」 제24조에 따른 경영상 이유에 의한 해고를 한 후 대통령령으로 정하는 기간이 지나기 전에는 해당 업무에 파견근로자를 사용하여서는 아니 된다.

=== 3 ===
metadata: {'article_no': '제40조', 'chapter_title': '근로계약', 'chapter_no': '제2장', 'law_name': '근로기준법', 'article_title': '취업 방해의 금지'}
content: 근로계약 취업 방해의 금지 누구든지 근로자의 취업을 방해할 목적으로 비밀 기호 또는 명부를 작성ㆍ사용하거나 통신 을 하여서는 아니 된다.

=== 4 ===
metadata: {'article_no': '제20조', 'law_name': '근로기준법', 'chapter_no': '', 'article_title': '근로자 명부의 기재사항', 'chapter_title': ''}
content: 근로자 명부의 기재사항 법 제41조

In [ ]:
docs = law_db.get()

names = set()

for m in docs["metadatas"]:
    names.add(m["law_name"])

for n in sorted(names):
    print(n)

근로기준법
근로자퇴직급여 보장법
기간제 및 단기근로자 보호등에 관한 법률
남녀고용평등과 일가정 양립 지원에 관한 법률
노동조합 및 노동관계조정법
산업안전보건법
최저임금법
파견근로자 보호 등에 관한 법률
